# 03 — KeyDiff Scoring on Flash Attention KV Caches

This notebook combines the Flash Attention gather primitive from notebook 02
with KeyDiff's actual key-similarity scoring, replacing the random token
selection used in the previous experiments.

The goal is to validate the end-to-end pipeline that the vLLM integration
will use: gather cached keys from the paged cache, score them with KeyDiff,
and make per-head keep/skip filtering decisions.

We run four experiments:
1. **Standalone scoring** — extract KeyDiff scoring into a pure function
   that works on vLLM-shaped tensors
2. **Score validation** — verify that our standalone scoring matches
   kvpress's `KeyDiffPress.score()` on the same input
3. **Gather + score pipeline** — scatter keys into a paged cache, gather
   them back, score them, and verify the scores are identical to scoring
   the original dense keys directly
4. **Filtering decisions** — simulate decode steps with per-head keep/skip
   decisions and validate against kvpress's `FilteringPress`

This completes step 2 of the Phase 2 roadmap (see
`feasibility/cache-filtering-approach.md`).

## Imports and Setup

In [ ]:
import torch
import torch.nn.functional as F
from vllm import _custom_ops as ops

torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
try:
    from kvpress import KeyDiffPress, FilteringPress
    HAS_KVPRESS = True
    print("kvpress available — will validate against it")
except ImportError:
    HAS_KVPRESS = False
    print("kvpress not installed — skipping cross-validation experiments")

## Configuration

In [ ]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
SEQ_LEN = 137
DEVICE = "cuda"

## Cache and Gather Functions

Reused from notebook 02 — Flash Attention layout
`[num_blocks, block_size, num_kv_heads, head_size]`.

In [ ]:
def create_kv_caches_flash(num_blocks, block_size, num_kv_heads, head_size,
                           dtype, device="cuda"):
    key_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    return key_cache, value_cache


def build_slot_mapping_for_positions(block_table, positions, block_size):
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping, block_size):
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size
    keys = key_cache[block_indices, offsets]
    values = value_cache[block_indices, offsets]
    return keys, values


print("Cache functions defined")

## Standalone KeyDiff Scoring

KeyDiff scores tokens by how similar their key vectors are to the average
key. Tokens with distinctive keys (dissimilar to the mean) get high scores
and are kept. Tokens with redundant keys (similar to the mean) get low
scores and are candidates for eviction.

The scoring is three operations:
1. L2-normalize all keys
2. Compute the anchor (mean of normalized keys)
3. Score = negative cosine similarity to the anchor

kvpress's `KeyDiffPress.score()` expects keys in shape
`[batch, num_kv_heads, seq_len, head_dim]`. Our standalone version works
on `[seq_len, num_kv_heads, head_dim]` — the shape returned by the Flash
Attention gather — and returns `[num_kv_heads, seq_len]`.

In [ ]:
def keydiff_score(keys, valid_lengths=None):
    """Score keys using KeyDiff's key-similarity metric.

    Parameters
    ----------
    keys : torch.Tensor
        Shape [seq_len, num_kv_heads, head_dim] — the vLLM gather output.
    valid_lengths : torch.Tensor, optional
        Shape [num_kv_heads] — per-head valid prefix length. If provided,
        the anchor is computed from valid positions only (for use with
        PaddedTensor-style ragged caches).

    Returns
    -------
    torch.Tensor
        Shape [num_kv_heads, seq_len]. Higher scores = more important.
    """
    # Transpose to [num_kv_heads, seq_len, head_dim] for per-head ops
    keys_by_head = keys.permute(1, 0, 2)
    normalized = F.normalize(keys_by_head, p=2, dim=-1)

    if valid_lengths is not None:
        # Masked mean: only valid positions contribute to the anchor
        seq_len = keys.shape[0]
        indices = torch.arange(seq_len, device=keys.device)
        mask = indices.unsqueeze(0) < valid_lengths.unsqueeze(1)  # [heads, seq_len]
        mask_f = mask.unsqueeze(-1).float()  # [heads, seq_len, 1]
        anchor = (
            (normalized * mask_f).sum(dim=1, keepdim=True)
            / mask_f.sum(dim=1, keepdim=True).clamp(min=1)
        )
    else:
        anchor = normalized.mean(dim=1, keepdim=True)

    scores = -F.cosine_similarity(keys_by_head, anchor, dim=-1)
    return scores  # [num_kv_heads, seq_len]


print("keydiff_score defined")

## Filtering Decision Logic

Given scores for all tokens (including the new one at the last position),
decide per head whether the new token survives. This mirrors
`FilteringPress.compress()`: compute `n_kept` from the total tokens seen
and the compression ratio, find the top-k threshold, and check whether
the new token's score is above it.

In [ ]:
def filter_new_token(scores, total_tokens_seen, compression_ratio):
    """Decide per head whether the newest token survives filtering.

    Parameters
    ----------
    scores : torch.Tensor
        Shape [num_kv_heads, seq_len]. All tokens scored, including the
        new one at position seq_len - 1.
    total_tokens_seen : int
        Logical count of all tokens processed so far (including skipped
        ones). Drives the n_kept calculation.
    compression_ratio : float
        Target fraction of tokens to filter out.

    Returns
    -------
    torch.Tensor
        Shape [num_kv_heads] — boolean. True = keep, False = reject.
    """
    n_kept = max(1, int(total_tokens_seen * (1 - compression_ratio)))
    n_kept = min(n_kept, scores.shape[-1])
    threshold = scores.topk(n_kept, dim=-1, sorted=True).values[:, -1]
    return scores[:, -1] >= threshold


print("filter_new_token defined")

## Experiment 1 — Standalone Scoring

Verify that `keydiff_score` produces sensible results: distinctive keys
should score higher than redundant ones.

In [ ]:
torch.manual_seed(42)
keys = torch.randn(SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)

scores = keydiff_score(keys)

print(f"Keys shape:   {keys.shape}")
print(f"Scores shape: {scores.shape}")
print(f"Score range:  [{scores.min().item():.4f}, {scores.max().item():.4f}]")
print(f"Score mean:   {scores.mean().item():.4f}")
print(f"Score std:    {scores.std().item():.4f}")

In [ ]:
# Sanity check: a key aligned with the anchor direction should score low
# (redundant), while one pointing opposite should score high (distinctive).
# Cosine similarity is scale-invariant, so we construct directions explicitly.

keys_by_head = keys.permute(1, 0, 2)  # [heads, seq_len, dim]
normalized = F.normalize(keys_by_head, p=2, dim=-1)
anchor = normalized.mean(dim=1)  # [heads, dim]

# Make the last token point in the anchor direction — maximally redundant
keys_redundant = keys.clone()
keys_redundant[-1] = anchor
scores_redundant = keydiff_score(keys_redundant)

# Make the last token point opposite to the anchor — maximally distinctive
keys_distinctive = keys.clone()
keys_distinctive[-1] = -anchor
scores_distinctive = keydiff_score(keys_distinctive)

print(f"Redundant token score  (last pos, head 0): {scores_redundant[0, -1].item():.4f}")
print(f"Random token score     (last pos, head 0): {scores[0, -1].item():.4f}")
print(f"Distinctive token score(last pos, head 0): {scores_distinctive[0, -1].item():.4f}")
print()

for h in range(NUM_KV_HEADS):
    assert scores_distinctive[h, -1] > scores_redundant[h, -1], (
        f"Head {h}: distinctive should score higher than redundant"
    )

print("Sanity check passed: -anchor scores higher than +anchor on all heads")

## Experiment 2 — Cross-Validation Against kvpress

Verify that our standalone `keydiff_score` produces the same scores as
kvpress's `KeyDiffPress.score()`. The only difference is tensor layout:
vLLM uses `[seq_len, heads, dim]`, kvpress uses `[batch, heads, seq_len, dim]`.

In [ ]:
if not HAS_KVPRESS:
    print("Skipping — kvpress not installed")
else:
    torch.manual_seed(42)
    keys_vllm = torch.randn(
        SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
    )

    # Our standalone scoring
    our_scores = keydiff_score(keys_vllm)  # [heads, seq_len]

    # kvpress scoring: reshape to [batch=1, heads, seq_len, dim]
    keys_kvpress = keys_vllm.permute(1, 0, 2).unsqueeze(0)
    press = KeyDiffPress()
    kvpress_scores = press.score(
        module=None,
        hidden_states=None,
        keys=keys_kvpress,
        values=None,
        attentions=None,
        kwargs={},
    )  # [batch=1, heads, seq_len]

    # Compare: our [heads, seq_len] vs kvpress [1, heads, seq_len]
    kvpress_scores_squeezed = kvpress_scores.squeeze(0)

    max_diff = (our_scores - kvpress_scores_squeezed).abs().max().item()
    print(f"Max score difference: {max_diff}")

    torch.testing.assert_close(
        our_scores, kvpress_scores_squeezed,
        atol=1e-4, rtol=1e-4,
    )
    print("Scores match kvpress KeyDiffPress.score()")

## Experiment 3 — Gather from Paged Cache + Score

The full pipeline for vLLM integration: keys live in the paged cache,
must be gathered into a dense buffer, then scored. Verify that scoring
gathered keys produces the same result as scoring the original dense keys.

In [ ]:
torch.manual_seed(42)
keys_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)
values_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)

# Create paged cache and scatter
num_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE + 4
key_cache, value_cache = create_kv_caches_flash(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

num_seq_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table = torch.arange(num_seq_blocks, dtype=torch.long, device=DEVICE)
positions = torch.arange(SEQ_LEN, dtype=torch.long, device=DEVICE)
slot_mapping = build_slot_mapping_for_positions(block_table, positions, BLOCK_SIZE)

k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
ops.reshape_and_cache_flash(
    keys_original, values_original,
    key_cache, value_cache,
    slot_mapping, "auto", k_scale, v_scale,
)

print(f"Scattered {SEQ_LEN} tokens into paged cache")

In [ ]:
# Gather back and score
keys_gathered, _ = gather_from_paged_cache(
    key_cache, value_cache, slot_mapping, BLOCK_SIZE,
)

scores_from_original = keydiff_score(keys_original)
scores_from_gathered = keydiff_score(keys_gathered)

max_diff = (scores_from_original - scores_from_gathered).abs().max().item()
print(f"Max score difference (original vs gathered): {max_diff}")

torch.testing.assert_close(
    scores_from_original, scores_from_gathered, atol=0, rtol=0,
)
print("Scoring gathered keys is identical to scoring original keys")

## Experiment 4 — Filtering Decisions

Simulate a sequence of decode steps. At each step:
1. Gather all cached keys from the paged cache
2. Append the new token's key
3. Score all keys with KeyDiff
4. Apply the filtering threshold to decide per head whether to keep
   the new token

We validate the keep/skip decisions against kvpress's `FilteringPress`
on the same key sequence.

In [ ]:
PREFILL_LEN = 64
NUM_DECODE_STEPS = 32
COMPRESSION_RATIO = 0.5

torch.manual_seed(123)
all_keys = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)

print(f"Prefill tokens:     {PREFILL_LEN}")
print(f"Decode steps:       {NUM_DECODE_STEPS}")
print(f"Compression ratio:  {COMPRESSION_RATIO}")
print(f"Total keys:         {all_keys.shape[0]}")

In [ ]:
# Simulate decode: prefill tokens are always cached, then each decode
# token is scored and either kept or skipped.

cached_keys = all_keys[:PREFILL_LEN].clone()  # all prefill tokens cached
decisions = []  # (step, per_head_keep) pairs

for step in range(NUM_DECODE_STEPS):
    new_key = all_keys[PREFILL_LEN + step]  # [num_kv_heads, head_dim]
    total_tokens_seen = PREFILL_LEN + step + 1

    # Append new token to cached keys for scoring
    keys_with_new = torch.cat(
        [cached_keys, new_key.unsqueeze(0)], dim=0,
    )  # [cache_len + 1, heads, dim]

    scores = keydiff_score(keys_with_new)
    keep_per_head = filter_new_token(scores, total_tokens_seen, COMPRESSION_RATIO)

    decisions.append({
        "step": step,
        "total_seen": total_tokens_seen,
        "cache_len_before": cached_keys.shape[0],
        "keep": keep_per_head.clone(),
        "all_keep": keep_per_head.all().item(),
        "any_keep": keep_per_head.any().item(),
    })

    # For simplicity, use uniform (all-or-nothing) caching here:
    # if ANY head wants to keep the token, cache it.
    # (The real FilteringPress uses per-head ragged lengths via PaddedTensor,
    # but for decision validation, the scoring and threshold logic is
    # what matters.)
    if keep_per_head.any():
        cached_keys = keys_with_new

kept_count = sum(1 for d in decisions if d["any_keep"])
skipped_count = NUM_DECODE_STEPS - kept_count
effective_ratio = skipped_count / (PREFILL_LEN + NUM_DECODE_STEPS)

print(f"\nResults:")
print(f"  Kept:    {kept_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Skipped: {skipped_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Final cache size: {cached_keys.shape[0]} tokens")
print(f"  Effective compression: {effective_ratio:.2%} of total tokens skipped")
print()

for d in decisions:
    heads_kept = d['keep'].sum().item()
    marker = "KEEP" if d['any_keep'] else "SKIP"
    print(
        f"  step {d['step']:2d}  seen={d['total_seen']:3d}  "
        f"cache={d['cache_len_before']:3d}  "
        f"heads={heads_kept}/{NUM_KV_HEADS}  {marker}"
    )

## Experiment 5 — Cross-Validation of Filtering Decisions Against kvpress

Run the same key sequence through kvpress's `FilteringPress` and verify
that the per-head keep/skip decisions match our standalone implementation.

This requires a mock `nn.Module` with a `layer_idx` attribute, and
`position_ids` in kwargs — the only external dependencies of
`FilteringPress.compress()`.

In [ ]:
if not HAS_KVPRESS:
    print("Skipping — kvpress not installed")
else:
    from types import SimpleNamespace

    mock_module = SimpleNamespace(layer_idx=0, head_dim=HEAD_SIZE)

    fp = FilteringPress(
        base_press=KeyDiffPress(),
        target_compression_ratio=COMPRESSION_RATIO,
    )

    # Replay the same key sequence through FilteringPress
    # Start with prefill keys in kvpress format [batch=1, heads, seq_len, dim]
    kvpress_keys = all_keys[:PREFILL_LEN].permute(1, 0, 2).unsqueeze(0).clone()
    kvpress_values = torch.zeros_like(kvpress_keys)  # values unused by KeyDiff scoring

    mismatches = 0
    fp.reset()

    for step in range(NUM_DECODE_STEPS):
        new_key = all_keys[PREFILL_LEN + step]  # [heads, dim]
        total_tokens_seen = PREFILL_LEN + step + 1

        # Append new token in kvpress format
        new_key_kvpress = new_key.unsqueeze(0).unsqueeze(0)  # [1, 1, heads, dim]
        new_key_kvpress = new_key_kvpress.permute(0, 2, 1, 3)  # [1, heads, 1, dim]
        keys_in = torch.cat([kvpress_keys, new_key_kvpress], dim=2)
        values_in = torch.cat(
            [kvpress_values, torch.zeros_like(new_key_kvpress)], dim=2,
        )

        position_ids = torch.arange(
            total_tokens_seen, device=DEVICE,
        ).unsqueeze(0)

        keys_out, values_out = fp.compress(
            module=mock_module,
            hidden_states=None,
            keys=keys_in,
            values=values_in,
            attentions=None,
            kwargs={"position_ids": position_ids},
        )

        # FilteringPress returns shorter keys if the token was rejected
        # by all heads (shrink), or same length if at least one head kept it.
        kvpress_kept = keys_out.shape[2] >= keys_in.shape[2]
        our_kept = decisions[step]["any_keep"]

        if kvpress_kept != our_kept:
            mismatches += 1
            print(
                f"  MISMATCH step {step}: ours={our_kept}, "
                f"kvpress={kvpress_kept}"
            )

        # Use kvpress output as input for next step
        kvpress_keys = keys_out
        kvpress_values = values_out

    if mismatches == 0:
        print(
            f"All {NUM_DECODE_STEPS} filtering decisions match "
            f"kvpress FilteringPress"
        )
    else:
        print(f"\n{mismatches}/{NUM_DECODE_STEPS} decisions differ")

## Experiment 6 — Full Pipeline on Paged Cache

End-to-end: keys live in the Flash Attention paged cache. At each decode
step, gather cached keys, score together with the new token, and decide
whether to write the new token to the cache.

In [ ]:
torch.manual_seed(123)
all_keys = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)
all_values = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)

total_len = PREFILL_LEN + NUM_DECODE_STEPS
num_blocks = (total_len + BLOCK_SIZE - 1) // BLOCK_SIZE + 4
key_cache, value_cache = create_kv_caches_flash(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

num_seq_blocks = (total_len + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table = torch.arange(num_seq_blocks, dtype=torch.long, device=DEVICE)
k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

# Scatter prefill tokens
prefill_positions = torch.arange(PREFILL_LEN, dtype=torch.long, device=DEVICE)
prefill_slots = build_slot_mapping_for_positions(
    block_table, prefill_positions, BLOCK_SIZE,
)
ops.reshape_and_cache_flash(
    all_keys[:PREFILL_LEN], all_values[:PREFILL_LEN],
    key_cache, value_cache,
    prefill_slots, "auto", k_scale, v_scale,
)

print(f"Prefilled {PREFILL_LEN} tokens into paged cache")

In [ ]:
# Simulate decode with filtering on the paged cache
cache_len = PREFILL_LEN  # tracks how many slots are used
paged_decisions = []

for step in range(NUM_DECODE_STEPS):
    total_tokens_seen = PREFILL_LEN + step + 1
    new_key = all_keys[PREFILL_LEN + step].unsqueeze(0)    # [1, heads, dim]
    new_value = all_values[PREFILL_LEN + step].unsqueeze(0)

    # 1. Gather cached keys from paged cache
    cached_positions = torch.arange(cache_len, dtype=torch.long, device=DEVICE)
    cached_slots = build_slot_mapping_for_positions(
        block_table, cached_positions, BLOCK_SIZE,
    )
    cached_keys, _ = gather_from_paged_cache(
        key_cache, value_cache, cached_slots, BLOCK_SIZE,
    )

    # 2. Score cached keys + new token together
    keys_with_new = torch.cat([cached_keys, new_key], dim=0)
    scores = keydiff_score(keys_with_new)

    # 3. Filtering decision
    keep_per_head = filter_new_token(scores, total_tokens_seen, COMPRESSION_RATIO)

    paged_decisions.append(keep_per_head.any().item())

    # 4. Write to cache only if kept
    if keep_per_head.any():
        new_position = torch.tensor([cache_len], dtype=torch.long, device=DEVICE)
        new_slot = build_slot_mapping_for_positions(
            block_table, new_position, BLOCK_SIZE,
        )
        ops.reshape_and_cache_flash(
            new_key, new_value,
            key_cache, value_cache,
            new_slot, "auto", k_scale, v_scale,
        )
        cache_len += 1

# Compare against the dense-tensor decisions from experiment 4
dense_decisions = [d["any_keep"] for d in decisions]

mismatches = sum(
    1 for p, d in zip(paged_decisions, dense_decisions) if p != d
)

print(f"Final cache size: {cache_len} tokens")
print(f"Kept: {sum(paged_decisions)}/{NUM_DECODE_STEPS} decode tokens")
print()

if mismatches == 0:
    print(
        f"All {NUM_DECODE_STEPS} decisions match between "
        f"paged-cache and dense-tensor pipelines"
    )
else:
    print(f"{mismatches}/{NUM_DECODE_STEPS} decisions differ")
    for i, (p, d) in enumerate(zip(paged_decisions, dense_decisions)):
        if p != d:
            print(f"  step {i}: paged={p}, dense={d}")

## Experiment 7 — Parametric Validation

Run the gather + score pipeline across multiple configurations to verify
the scoring is robust to different dtypes, head counts, and head sizes.

In [ ]:
import itertools

DTYPES = [torch.bfloat16, torch.float16]
KV_HEADS = [4, 8]
HEAD_SIZES = [64, 128]
COMP_RATIOS = [0.25, 0.5]

passed = 0
failed = 0

for dtype, nkv, hs in itertools.product(DTYPES, KV_HEADS, HEAD_SIZES):
    seq_len = 137
    bs = 16
    nb = (seq_len + bs - 1) // bs + 4

    kc, vc = create_kv_caches_flash(nb, bs, nkv, hs, dtype, DEVICE)

    torch.manual_seed(42)
    keys_orig = torch.randn(seq_len, nkv, hs, dtype=dtype, device=DEVICE)
    vals_orig = torch.randn(seq_len, nkv, hs, dtype=dtype, device=DEVICE)

    nsb = (seq_len + bs - 1) // bs
    bt = torch.arange(nsb, dtype=torch.long, device=DEVICE)
    pos = torch.arange(seq_len, dtype=torch.long, device=DEVICE)
    sm = build_slot_mapping_for_positions(bt, pos, bs)

    ks = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
    vs = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
    ops.reshape_and_cache_flash(keys_orig, vals_orig, kc, vc, sm, "auto", ks, vs)

    # Gather and score
    kg, _ = gather_from_paged_cache(kc, vc, sm, bs)
    scores_orig = keydiff_score(keys_orig)
    scores_gathered = keydiff_score(kg)

    try:
        torch.testing.assert_close(scores_orig, scores_gathered, atol=0, rtol=0)
        passed += 1
    except Exception as e:
        failed += 1
        print(f"FAIL score: dtype={dtype}, heads={nkv}, hs={hs}: {e}")

    # Filtering decision test
    for cr in COMP_RATIOS:
        keep = filter_new_token(scores_orig, seq_len, cr)
        keep_gathered = filter_new_token(scores_gathered, seq_len, cr)

        try:
            torch.testing.assert_close(keep, keep_gathered)
            passed += 1
        except Exception as e:
            failed += 1
            print(
                f"FAIL filter: dtype={dtype}, heads={nkv}, "
                f"hs={hs}, cr={cr}: {e}"
            )

total = passed + failed
print(f"\n{passed}/{total} tests passed")
if failed == 0:
    print("All configurations verified")

## Notes and Next Steps

**Scoring validated.** `keydiff_score` produces identical results to
kvpress's `KeyDiffPress.score()` and is invariant to the
scatter/gather round trip through the paged cache.

**Filtering decisions validated.** The standalone `filter_new_token`
function makes the same per-head keep/skip decisions as kvpress's
`FilteringPress`, and these decisions are identical whether scoring
from dense tensors or from gathered paged cache data.

**What this enables:** With scoring and filtering validated on the
Flash Attention cache layout, the next step is integrating this into
vLLM's `Attention.forward()` — step 3 of the Phase 2 roadmap. The
integration will:
1. Gather cached keys after `do_kv_cache_update`
2. Score with `keydiff_score`
3. Filter the slot mapping with `filter_new_token`
4. Track logical positions for RoPE separately from cache positions